In [7]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob
from pathlib import Path
import os
import math
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from torch.cuda.amp import GradScaler, autocast

from transformers import (
    BertTokenizer,
    BertModel,
    get_linear_schedule_with_warmup,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    precision_recall_curve,
    roc_curve,
)

warnings.filterwarnings("ignore")

In [8]:
SEED           = 42
DEVICE         = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_NAME     = "bert-base-uncased"   # swap for "roberta-base" if desired
MAX_LENGTH     = 512                   # Baseline: truncation strategy
CHUNK_SIZE     = 512                   # Advanced: chunk size for sliding-window
CHUNK_OVERLAP  = 64                    # tokens of overlap between chunks
BATCH_SIZE     = 16                    # reduce to 8 if OOM on GPU
EPOCHS         = 3
LR             = 2e-5
WARMUP_RATIO   = 0.1
DROPOUT        = 0.1
WEIGHT_DECAY   = 0.01
GRAD_CLIP      = 1.0
THRESHOLD      = 0.5                   # classification threshold (tunable)
OUTPUT_DIR     = Path("./bert_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

def set_seed(seed: int = SEED):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()


In [9]:
pd.set_option('display.max_columns', None)

# Download dataset and get path
dataset_path = kagglehub.dataset_download("saurabhshahane/fake-news-classification")

# Find CSV file(s) in the dataset
csv_files = list(Path(dataset_path).rglob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"No CSV files found in {dataset_path}")

# Load first CSV (or combine multiple if needed, e.g. Fake.csv + True.csv)
# index_col=0 skips the "Unnamed: 0" column (saved index from original CSV)
if len(csv_files) == 1:
    df = pd.read_csv(csv_files[0], index_col=0)
else:
    # If dataset has separate Fake.csv and True.csv, load and combine
    dfs = []
    for f in csv_files:
        dfs.append(pd.read_csv(f, index_col=0))
    df = pd.concat(dfs, ignore_index=True)

print(f"Loaded {len(df)} rows from {[f.name for f in csv_files]}")
df.head()


Loaded 72134 rows from ['WELFake_Dataset.csv']


,title,text,label
0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,NaN,Did they post their votes for Hillary already?,1
2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1
3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1


In [12]:
df_cleaned = df.dropna(subset=["title", "text"]).copy()
print(f"After null drop : {len(df_cleaned):,}")

After null drop : 71,537


In [11]:
df_cleaned['label'].value_counts(normalize=True)

label
1    0.510351
0    0.489649
Name: proportion, dtype: float64

In [13]:
df_cleaned.drop_duplicates(subset=["title", "text"], inplace=True)
print(f"After dedup      : {len(df_cleaned):,}")

After dedup      : 63,121


In [14]:
print("\nLabel distribution (0=Real, 1=Fake):")
print(df_cleaned["label"].value_counts(normalize=True).round(4))


Label distribution (0=Real, 1=Fake):
label
0    0.5512
1    0.4488
Name: proportion, dtype: float64


In [15]:
# Concatenate title + [SEP] + text so BERT can model
# the cross-attention between headline and body text.
df_cleaned["full_content"] = (
    df_cleaned["title"].str.strip()
    + " [SEP] "
    + df_cleaned["text"].str.strip()
)

In [16]:
# Stratified Train / Val / Test Split 
y = df_cleaned["label"].values
X = df_cleaned["full_content"].values

# 70% Train | 10% Val | 20% Test — stratified throughout
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.125, random_state=SEED, stratify=y_temp
)

print(f"\nSplit sizes → Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")
print(f"Fake ratio  → Train: {y_train.mean():.3f} | Val: {y_val.mean():.3f} | Test: {y_test.mean():.3f}")



Split sizes → Train: 44,184 | Val: 6,312 | Test: 12,625
Fake ratio  → Train: 0.449 | Val: 0.449 | Test: 0.449


---
# BERT-Specific Preprocessing

In [17]:
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

In [18]:
# Baseline Dataset (Truncation to 512 tokens)
class FakeNewsDatasetTrunc(Dataset):
    """
    Tokenizes on-the-fly.
    Strategy A — Truncation: articles > 512 tokens are cut at the
    512-token boundary, preserving the 'lead' (most informative part).
    """
    def __init__(self, texts, labels, tokenizer, max_len=MAX_LENGTH):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            # Input IDs   : token index in BERT vocabulary
            "input_ids"      : encoding["input_ids"].squeeze(0),
            # Attention Mask : 1 for real tokens, 0 for padding
            "attention_mask" : encoding["attention_mask"].squeeze(0),
            # Token Type IDs : 0 for all tokens (title+text treated as one)
            "token_type_ids" : encoding["token_type_ids"].squeeze(0),
            "label"          : torch.tensor(self.labels[idx], dtype=torch.float),
        }


In [19]:
# Advanced Dataset (Sliding-Window Chunks) 
# Used ONLY for TEST-SET inference ("chunking strategy").
# The trained model votes across all chunks of an article.
class FakeNewsDatasetChunked(Dataset):
    """
    Strategy B — Sliding Window:
    Breaks each article into overlapping chunks of CHUNK_SIZE tokens
    with CHUNK_OVERLAP tokens of context carried forward.
    Inference aggregates chunk-level probabilities (majority vote / mean).
    """
    def __init__(self, texts, labels, tokenizer,
                 chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
        self.tokenizer  = tokenizer
        self.chunk_size = chunk_size
        self.overlap    = overlap
        self.samples    = []  # list of (input_ids, mask, type_ids, label, article_idx)

        for art_idx, (text, label) in enumerate(zip(texts, labels)):
            encoding = tokenizer(
                text,
                add_special_tokens=False,
                return_tensors="pt",
            )
            ids   = encoding["input_ids"].squeeze(0)
            masks = encoding["attention_mask"].squeeze(0)
            types = torch.zeros_like(ids)

            stride  = chunk_size - overlap - 2   # -2 for [CLS] and [SEP]
            total   = ids.size(0)
            starts  = list(range(0, max(total - stride, 1), stride))

            for start in starts:
                end   = min(start + chunk_size - 2, total)
                chunk_ids   = torch.cat([
                    torch.tensor([tokenizer.cls_token_id]),
                    ids[start:end],
                    torch.tensor([tokenizer.sep_token_id]),
                ])
                chunk_masks = torch.cat([
                    torch.ones(1, dtype=torch.long),
                    masks[start:end],
                    torch.ones(1, dtype=torch.long),
                ])
                chunk_types = torch.zeros_like(chunk_ids)

                # Pad to chunk_size
                pad_len = chunk_size - chunk_ids.size(0)
                chunk_ids   = torch.cat([chunk_ids,   torch.zeros(pad_len, dtype=torch.long)])
                chunk_masks = torch.cat([chunk_masks, torch.zeros(pad_len, dtype=torch.long)])
                chunk_types = torch.cat([chunk_types, torch.zeros(pad_len, dtype=torch.long)])

                self.samples.append({
                    "input_ids"      : chunk_ids,
                    "attention_mask" : chunk_masks,
                    "token_type_ids" : chunk_types,
                    "label"          : torch.tensor(label, dtype=torch.float),
                    "article_idx"    : art_idx,
                })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

In [24]:
# Build DataLoaders
train_dataset = FakeNewsDatasetTrunc(X_train, y_train, tokenizer)
val_dataset   = FakeNewsDatasetTrunc(X_val,   y_val,   tokenizer)
test_dataset  = FakeNewsDatasetTrunc(X_test,  y_test,  tokenizer)  # baseline test

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}")

Train batches: 2762 | Val batches: 395 | Test batches: 790


---
# Model Architecture

In [25]:
class BERTFakeNewsClassifier(nn.Module):
    """
    Architecture:
      bert-base-uncased  (pre-trained)       
      → 768-dim [CLS] representation         

      Dropout(0.1)                           
      Linear(768 → 1)                        
      (raw logit — BCEWithLogitsLoss)        

    """
    def __init__(self, model_name: str = MODEL_NAME,
                 dropout: float = DROPOUT,
                 freeze_bert: bool = False):
        super().__init__()
        self.bert    = BertModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.bert.config.hidden_size, 1)

        # Optional: freeze all BERT layers (fast training, lower accuracy)
        # Set freeze_bert=True to use; False for full fine-tuning (recommended)
        if freeze_bert:
            for param in self.bert.parameters():
                param.requires_grad = False
            print("NOTE: BERT base layers are FROZEN (only head is trained).")
        else:
            print("Fine-tuning ENTIRE network (BERT + classification head).")

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
        )
        # [CLS] token is the pooled summary of the whole sequence
        cls_output = outputs.pooler_output          # shape: (batch, 768)
        cls_output = self.dropout(cls_output)
        logits     = self.classifier(cls_output)    # shape: (batch, 1)
        return logits.squeeze(-1)                   # shape: (batch,)


model = BERTFakeNewsClassifier(freeze_bert=False).to(DEVICE)
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,}")


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1926.69it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Fine-tuning ENTIRE network (BERT + classification head).
Total parameters     : 109,483,009
Trainable parameters : 109,483,009


---
# Training Strategy and Optimization

In [26]:
# BCEWithLogitsLoss = Sigmoid + BCE combined in one numerically
# stable operation. Operates on raw logits (no sigmoid needed in forward()).
loss_fn = nn.BCEWithLogitsLoss()

# Optimizer (AdamW) 
no_decay  = ["bias", "LayerNorm.weight"]
param_groups = [
    {"params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
     "weight_decay": WEIGHT_DECAY},
    {"params": [p for n, p in model.named_parameters() if     any(nd in n for nd in no_decay)],
     "weight_decay": 0.0},
]
optimizer = AdamW(param_groups, lr=LR)

# Linear Warmup + Decay Scheduler
total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler    = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

# Mixed Precision (AMP)
scaler = GradScaler(enabled=DEVICE.type == "cuda")

print(f"Total training steps : {total_steps:,}")
print(f"Warmup steps         : {warmup_steps:,}")
print(f"Device               : {DEVICE}")

Total training steps : 8,286
Warmup steps         : 828
Device               : cpu


In [27]:
# Training & Validation Loop
def train_epoch(model, loader, optimizer, scheduler, loss_fn, scaler):
    model.train()
    total_loss, all_preds, all_labels = 0.0, [], []

    for batch in loader:
        input_ids   = batch["input_ids"].to(DEVICE)
        attn_mask   = batch["attention_mask"].to(DEVICE)
        type_ids    = batch["token_type_ids"].to(DEVICE)
        labels      = batch["label"].to(DEVICE)

        optimizer.zero_grad()

        with autocast(enabled=DEVICE.type == "cuda"):
            logits = model(input_ids, attn_mask, type_ids)
            loss   = loss_fn(logits, labels)

        scaler.scale(loss).backward()
        # Gradient Clipping — prevents exploding gradients in transformers
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item()
        probs = torch.sigmoid(logits).detach().cpu().numpy()
        all_preds.extend((probs >= THRESHOLD).astype(int).tolist())
        all_labels.extend(labels.cpu().numpy().astype(int).tolist())

    avg_loss = total_loss / len(loader)
    acc      = accuracy_score(all_labels, all_preds)
    f1       = f1_score(all_labels, all_preds)
    return avg_loss, acc, f1


def eval_epoch(model, loader, loss_fn):
    model.eval()
    total_loss, all_probs, all_labels = 0.0, [], []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(DEVICE)
            attn_mask = batch["attention_mask"].to(DEVICE)
            type_ids  = batch["token_type_ids"].to(DEVICE)
            labels    = batch["label"].to(DEVICE)

            logits    = model(input_ids, attn_mask, type_ids)
            loss      = loss_fn(logits, labels)

            total_loss += loss.item()
            probs = torch.sigmoid(logits).cpu().numpy()
            all_probs.extend(probs.tolist())
            all_labels.extend(labels.cpu().numpy().astype(int).tolist())

    avg_loss = total_loss / len(loader)
    preds    = (np.array(all_probs) >= THRESHOLD).astype(int)
    acc      = accuracy_score(all_labels, preds)
    f1       = f1_score(all_labels, preds)
    auc      = roc_auc_score(all_labels, all_probs)
    return avg_loss, acc, f1, auc, np.array(all_probs), np.array(all_labels)


history = {"train_loss": [], "val_loss": [], "train_f1": [], "val_f1": [], "val_auc": []}
best_val_f1    = 0.0
best_model_path = OUTPUT_DIR / "bert_best_model.pt"

for epoch in range(1, EPOCHS + 1):
    print(f"\n── Epoch {epoch}/{EPOCHS} ──────────────────────────────────")
    t_loss, t_acc, t_f1 = train_epoch(model, train_loader, optimizer, scheduler, loss_fn, scaler)
    v_loss, v_acc, v_f1, v_auc, val_probs, val_labels = eval_epoch(model, val_loader, loss_fn)

    history["train_loss"].append(t_loss)
    history["val_loss"].append(v_loss)
    history["train_f1"].append(t_f1)
    history["val_f1"].append(v_f1)
    history["val_auc"].append(v_auc)

    print(f"  Train → Loss: {t_loss:.4f} | Acc: {t_acc:.4f} | F1: {t_f1:.4f}")
    print(f"  Val   → Loss: {v_loss:.4f} | Acc: {v_acc:.4f} | F1: {v_f1:.4f} | AUC: {v_auc:.4f}")

    # Save checkpoint if best validation F1
    if v_f1 > best_val_f1:
        best_val_f1 = v_f1
        torch.save(model.state_dict(), best_model_path)
        print(f"  ✓ Best model saved (val F1={best_val_f1:.4f})")

print(f"\nTraining complete. Best Val F1: {best_val_f1:.4f}")



── Epoch 1/3 ──────────────────────────────────


KeyboardInterrupt: 

In [ ]:
# Load best checkpoint
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))

# Baseline Test Evaluation (Truncation)
_, test_acc, test_f1, test_auc, test_probs, test_labels = eval_epoch(
    model, test_loader, loss_fn
)
test_preds = (test_probs >= THRESHOLD).astype(int)
print(f"\n[BASELINE — Truncation Strategy]")
print(f"  Test Accuracy : {test_acc:.4f}")
print(f"  Test F1       : {test_f1:.4f}")
print(f"  Test ROC-AUC  : {test_auc:.4f}")
print("\nClassification Report:")
print(classification_report(test_labels, test_preds, target_names=["Real", "Fake"]))

# Advanced Test Evaluation (Chunked Sliding-Window)
print("\n[ADVANCED — Sliding-Window Chunking Strategy]")
chunked_test_dataset = FakeNewsDatasetChunked(X_test, y_test, tokenizer)
chunked_test_loader  = DataLoader(chunked_test_dataset, batch_size=BATCH_SIZE,
                                  shuffle=False, num_workers=0)

model.eval()
chunk_probs_by_article = {}   # article_idx → list of chunk probabilities

with torch.no_grad():
    for batch in chunked_test_loader:
        input_ids   = batch["input_ids"].to(DEVICE)
        attn_mask   = batch["attention_mask"].to(DEVICE)
        type_ids    = batch["token_type_ids"].to(DEVICE)
        art_indices = batch["article_idx"]

        logits = model(input_ids, attn_mask, type_ids)
        probs  = torch.sigmoid(logits).cpu().numpy()

        for i, art_idx in enumerate(art_indices):
            art_idx = art_idx.item()
            chunk_probs_by_article.setdefault(art_idx, []).append(probs[i])

# Voting Mechanism: average chunk probabilities per article
n_articles      = len(X_test)
agg_probs       = np.zeros(n_articles)
for art_idx, probs_list in chunk_probs_by_article.items():
    agg_probs[art_idx] = np.mean(probs_list)   # "average probability" vote

chunk_preds   = (agg_probs >= THRESHOLD).astype(int)
chunk_labels  = y_test.astype(int)

chunk_acc  = accuracy_score(chunk_labels, chunk_preds)
chunk_f1   = f1_score(chunk_labels, chunk_preds)
chunk_auc  = roc_auc_score(chunk_labels, agg_probs)

print(f"  Test Accuracy : {chunk_acc:.4f}")
print(f"  Test F1       : {chunk_f1:.4f}")
print(f"  Test ROC-AUC  : {chunk_auc:.4f}")
print("\nClassification Report:")
print(classification_report(chunk_labels, chunk_preds, target_names=["Real", "Fake"]))
